# Rafiq GPU Job — توليد مشهد من حزمة الاستوديو

1. في Rafiq AI Studio افتح المشهد واضغط **تجهيز المشهد لجهاز GPU** ثم نزّل ZIP.
2. افتح هذا الدفتر في Colab واستخدم GPU مجانيًا إن توفر. تحقّق من خطة حسابك أولًا؛ الدفتر لا يستطيع معرفة ما إذا كانت الجلسة تخصم وحدات مدفوعة. لا تشترِ موارد أو وحدات حوسبة.
3. شغّل الخلايا بالترتيب وارفع **حزمة الطلب**، لا حزمة نتيجة قديمة.
4. بعد التوليد نزّل حزمة النتيجة واستوردها من **الإنتاج والتصدير** في الاستوديو. راجع اللقطة قبل اعتمادها.

تنتقل إلى جلسة Colab بيانات هذه اللقطة والصورة المرجعية المختارة. لا تُرفع مكتبة الشخصيات أو عينات الأصوات أو الكتاب كاملًا. لا يربط الدفتر Google Drive ولا ينشر خادمًا أو نفقًا عامًا؛ التفاعل كله داخل الخلايا.

**اختبار البداية:** صدّر لقطة ثانية واحدة بدقة 320×192 و8 خطوات. ينفذ الدفتر إعدادات الحزمة نفسها دون تغييرها. نقل الحزمة اختُبر بمحاكاة بروتوكول المحرك وملفات MP4 فعلية. نجح العامل المشترك مع Wan 2.1 على T4 في تجربة منفصلة؛ دورة الحزمة الكاملة على GPU ما زالت تحتاج اختبار قبول.

[سياسة وحدود Colab](https://research.google.com/colaboratory/faq.html)


In [ ]:
import json, os, shutil, subprocess, sys, time
from pathlib import Path
assert Path('/content').is_dir(), 'Open this notebook in Google Colab.'
assert shutil.which('nvidia-smi'), 'No free NVIDIA GPU is attached. Stop; do not buy compute units.'
subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'], check=True)
import torch
assert torch.cuda.is_available(), 'CUDA is not available in this runtime.'


## تحميل كود Rafiq

تثبيت في الجلسة المؤقتة فقط. إذا كان المجلد موجودًا من اختبار سابق، يُحدَّث فقط عندما يكون الكود نظيفًا بلا تعديلات محلية.


In [ ]:
repo = Path('/content/Rafiq-AI-Studio')
source_ref = 'feature/rafiq-studio-mvp'
source_url = 'https://github.com/ism87jo/Rafiq-AI-Studio.git'
if not repo.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', source_ref, source_url, str(repo)], check=True)
else:
    origin = subprocess.check_output(['git', '-C', str(repo), 'remote', 'get-url', 'origin'], text=True).strip()
    assert origin == source_url, 'The existing folder is not the expected Rafiq repository.'
    assert not subprocess.check_output(['git', '-C', str(repo), 'status', '--porcelain'], text=True).strip(), 'Existing source has local edits; preserve them before updating.'
    subprocess.run(['git', '-C', str(repo), 'fetch', 'origin', source_ref], check=True)
    subprocess.run(['git', '-C', str(repo), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
source_commit = subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True).strip()
print('Source commit:', source_commit)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(repo / 'requirements.txt'),
                '-r', str(repo / 'rafiq/studio/requirements.txt'), 'huggingface_hub'], check=True)
sys.path.insert(0, str(repo))


## رفع حزمة المشهد ومراجعة إعداداتها

هذه الخلية تفحص الحزمة ولا تولّد فيديو. لا يقبل المنفذ مخطط عُقد مخصصًا من ZIP؛ يعيد بناء مسار Wan المضمّن من إعدادات المشهد.


In [ ]:
from google.colab import files
from rafiq.studio.portable import inspect_request
from rafiq.studio.workflows import PRESETS
uploaded = files.upload()
assert len(uploaded) == 1, 'Upload exactly one request ZIP.'
content = next(iter(uploaded.values()))
assert len(content) <= 32 * 1024**2, 'Request ZIP is too large; upload a Rafiq scene request.'
request_path = Path('/content/rafiq-input.zip')
request_path.write_bytes(content)
request = inspect_request(request_path)
settings = request['project']['settings']
preset = PRESETS[settings['preset']]
print('Scene:', request['project']['scenes'][0]['title'])
print('Settings:', json.dumps(settings, ensure_ascii=False, indent=2))
print('Seed:', request['project']['scenes'][0]['seed'])


## تنزيل الأوزان المطلوبة وتسجيل بصماتها

ثلاثة ملفات فقط من مستودعات Comfy-Org الرسمية. تُثبت مراجعة كل مستودع عند التنزيل وتُسجّل SHA‑256 لكل ملف. يدعم المسار Wan 2.1 1.3B وWan 2.2 5B حسب الحزمة، دون API فيديو مدفوع.


In [ ]:
import hashlib
from huggingface_hub import HfApi, hf_hub_download
repo21 = 'Comfy-Org/Wan_2.1_ComfyUI_repackaged'
repo22 = 'Comfy-Org/Wan_2.2_ComfyUI_Repackaged'
video_repo = repo22 if settings['preset'] == 'wan22-5b' else repo21
wanted = [(video_repo, 'diffusion_models', preset['model']),
          (repo21, 'text_encoders', preset['encoder']),
          (video_repo, 'vae', preset['vae'])]
revisions, model_records = {}, []
for model_repo, category, filename in wanted:
    if model_repo not in revisions:
        revisions[model_repo] = HfApi().model_info(model_repo).sha
    revision = revisions[model_repo]
    cached = Path(hf_hub_download(model_repo, filename=f'split_files/{category}/{filename}',
                                  revision=revision, cache_dir='/content/rafiq-hf-cache'))
    target = repo / 'models' / category / filename
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.exists():
        assert target.resolve() == cached.resolve(), f'A different model already exists at {target}'
    else:
        target.symlink_to(cached)
    with target.open('rb') as stream:
        digest = hashlib.file_digest(stream, 'sha256').hexdigest()
    model_records.append(dict(repo=model_repo, revision=revision, file=f'{category}/{filename}', sha256=digest))
manifest = Path('/content/rafiq-gpu-model-manifest.json')
manifest.write_text(json.dumps(dict(source_commit=source_commit, models=model_records), indent=2))
print(manifest.read_text())


## تشغيل ComfyUI داخل الجلسة

يظل المحرك على loopback. هذه ليست استضافة عامة للاستوديو.


In [ ]:
import urllib.request

engine_log = Path('/content/rafiq-comfy.log')
if 'engine_process' not in globals() or engine_process.poll() is not None:
    with engine_log.open('wb') as log:
        engine_process = subprocess.Popen(
            [sys.executable, 'main.py', '--offline', '--listen', '127.0.0.1', '--port', '8188',
             '--disable-auto-launch', '--disable-all-custom-nodes', '--lowvram'],
            cwd=repo, stdout=log, stderr=subprocess.STDOUT,
            env=dict(os.environ, HF_HUB_OFFLINE='1', TRANSFORMERS_OFFLINE='1', HF_HUB_DISABLE_TELEMETRY='1'))
engine_ready = False
for attempt in range(180):
    if engine_process.poll() is not None:
        break
    try:
        with urllib.request.urlopen('http://127.0.0.1:8188/system_stats', timeout=2) as response:
            stats = json.load(response)
        print(json.dumps(stats, indent=2))
        engine_ready = True
        break
    except (OSError, TimeoutError):
        time.sleep(1)
if not engine_ready:
    print(engine_log.read_text(errors='replace')[-8000:])
    raise RuntimeError('ComfyUI did not start. No generation was submitted.')

sys.path.insert(0, str(repo))
from rafiq.smoke import verify_native_video
verify_native_video('http://127.0.0.1:8188', Path('/content'))


## تنفيذ المشهد

تشغيل هذه الخلية يبدأ التوليد الفعلي. يحتفظ المنفذ بحالة المهمة في مجلد عمل خاص داخل الجلسة. إذا انقطع التوليد أو فشل بعد إرسال الطلب فلن يعيد إرساله تلقائيًا؛ افحص سجل ComfyUI أولًا. إذا نجح التوليد وفشل التنزيل/التغليف، يمكن إعادة التغليف من الناتج المحفوظ دون توليد جديد.


In [ ]:
output = Path(f"/content/rafiq-gpu-result-{request['id']}.zip")
if output.exists():
    print('Result already exists. Use the download cell below; no generation was started.')
else:
    command = [sys.executable, '-m', 'rafiq.studio.portable', str(request_path), '--run', '--output', str(output),
               '--model-manifest', str(manifest), '--source-commit', source_commit]
    result = subprocess.run(command, cwd=repo, capture_output=True, text=True, timeout=2100)
    print(result.stdout)
    if result.returncode:
        print(result.stderr[-5000:])
        print(engine_log.read_text(errors='replace')[-8000:])
        raise RuntimeError('GPU render did not pass. Keep the work folder and inspect the saved prompt before retrying.')
assert output.is_file(), 'No completed result ZIP was created.'


## تنزيل النتيجة وإيقاف المحرك

استورد ZIP كما هو في الاستوديو. يحتفظ الاستوديو بتاريخ الطلب الأصلي، ويتحقق من بصمات الحزمة والفيديو، ولا يستبدل تعديلات المشهد تلقائيًا. زمن التنفيذ مُسجّل؛ التكلفة المالية لجهاز GPU لا تُقاس تلقائيًا.


In [ ]:
files.download(str(output))


In [ ]:
if 'engine_process' in globals() and engine_process.poll() is None:
    engine_process.terminate()
    engine_process.wait(timeout=20)
print('Engine stopped. Save the result, then use Runtime → Disconnect and delete runtime.')
